In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [ ]:
from src.ingestion.sidra_mapping import SidraCrops, SidraTable, SidraPeriod, SidraLocality, SidraVariables

sidra_variables_dict = {v.value: v.name.lower() for v in SidraVariables}
sidra_variables_dict.keys()
",".join(v.value for v in SidraVariables)

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import requests
import json
import sys
from src.ingestion.sidra_mapping import (
    BASE_URL_API_SIDRA,
    SidraCrops,
    SidraTable,
    SidraPeriod,
    SidraLocality,
    SidraVariables,
)
import pandas as pd

sys.path

# BASE_URL = "https://apisidra.ibge.gov.br/values"

table = SidraTable.CULTIVARS_PRODUCAO.value
period = SidraPeriod.P2010_2024.value
variables = ",".join(v.value for v in SidraVariables)
location = SidraLocality.PARANA.value
soybean_code = SidraCrops.SOJA.value
# tabela = "5457"
# periodo = "2010-2024"
# variaveis = "8331,216,214,112,215"
# localidade = "in n3 41"
# crop_code = "40124" # Soja

# url = f"{BASE_URL}/t/{tabela}/p/{periodo}/v/{variaveis}/n6/{localidade}/c782/{crop_code}"
url = f"{BASE_URL_API_SIDRA}/t/{table}/p/{period}/v/{variables}/n6/{location}/c782/{soybean_code}"

print(f"URL: {url}")

try:
    response = requests.get(url, timeout=60)
    print(f"Status Code: {response.status_code}")
    if response.status_code == 200:
        data = response.json()
        print("Data size:", len(data))
        print("First 2 elements:")
        print(json.dumps(data[:2], indent=4, ensure_ascii=False))
except Exception as e:
    print("Error:", e)

In [ ]:
import pandas as pd
from pathlib import Path

# BASE_DIR = Path(__file__).resolve().parent.parent
BASE_DIR = Path.cwd().parent

SAVE_DIR = BASE_DIR / "data" / "raw"
# SAVE_DIR = Path("../data/raw")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
SAVE_PATH = SAVE_DIR / "pam_parana_soybean_raw.parquet"
print(SAVE_PATH)
df = pd.DataFrame(data)
df.to_parquet(SAVE_PATH, engine="pyarrow", compression="snappy", index=False)

In [ ]:
print(json.dumps(data[:10], indent=4, ensure_ascii=False))

In [ ]:
import sys
from pathlib import Path

# Adiciona a raiz do projeto para o Python encontrar a pasta 'src'
sys.path.append(str(Path.cwd().parent))

from src.utils.logging_config import setup_logging
from src.ingestion.client import SidraClient
from src.ingestion.pipeline import IngestionPipeline

# Configura o logger
setup_logging()

# Inicializa o cliente e o pipeline
client = SidraClient()
pipeline = IngestionPipeline(client=client, base_dir=Path.cwd().parent)

# Roda a ingestão das 3 culturas
pipeline.run()